# Disagreement-Aware Probabilistic U-Net on LIDC

Selected paper: Kohl et al., *A Probabilistic U-Net for Segmentation of Ambiguous Images*, NeurIPS 2018.

Extension: predict the human inter-rater disagreement map from the image and align stochastic sample diversity with the pixels where annotators disagree.

## Dependencies

In [ ]:
import importlib
import subprocess
import sys

# Colab already provides NumPy, PyTorch, Pillow, Matplotlib, and tqdm. Avoid
# changing NumPy inside a live runtime; compiled packages can otherwise disagree
# about the NumPy ABI and raise "numpy.dtype size changed".
required = {
    "torch": "torch>=2.0",
    "numpy": None,
    "PIL": "Pillow>=10",
    "matplotlib": "matplotlib>=3.9",
    "tqdm": "tqdm>=4.66",
}
missing = []
for module_name, package_spec in required.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        if package_spec is None:
            raise
        missing.append(package_spec)

if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import numpy as np
import torch
print("numpy", np.__version__)
print("torch", torch.__version__)

## Configuration

In [ ]:
import csv
import json
import math
import os
import random
import shutil
import sys
import tarfile
import tempfile
import time
import urllib.request
from pathlib import Path
from types import SimpleNamespace

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch import distributions
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from IPython.display import Image as DisplayImage, Markdown, display


RUN_MODE = "smoke"  # "smoke", "pilot", or "final"
FORCE_RETRAIN = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("data/lidc")
OUTPUT_ROOT = Path("outputs_self_contained") / RUN_MODE
SEED = 1

MODE_CONFIGS = {
    "smoke": {
        "steps": 2,
        "max_train": 8,
        "max_val": 4,
        "max_test": 4,
        "batch_size": 1,
        "eval_batch_size": 1,
        "feature_maps": 2,
        "latent_size": 2,
        "depth": 2,
        "train_samples": 2,
        "eval_samples": 2,
        "eval_every": 1,
        "save_every": 1,
        "figure_cases": 2,
        "figure_samples": 2,
    },
    "pilot": {
        "steps": 10000,
        "max_train": None,
        "max_val": 256,
        "max_test": 256,
        "batch_size": 16,
        "eval_batch_size": 8,
        "feature_maps": 16,
        "latent_size": 6,
        "depth": 4,
        "train_samples": 4,
        "eval_samples": 16,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
    "final": {
        "steps": 100000,
        "max_train": None,
        "max_val": None,
        "max_test": None,
        "batch_size": 32,
        "eval_batch_size": 8,
        "feature_maps": 32,
        "latent_size": 6,
        "depth": 5,
        "train_samples": 4,
        "eval_samples": 16,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
}
CFG = MODE_CONFIGS[RUN_MODE]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAIN_VARIANTS = ("baseline", "head", "full")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"mode={RUN_MODE} device={DEVICE} output={OUTPUT_ROOT}")

## LIDC Download

In [ ]:
BASE_URL = "https://storage.googleapis.com/hpunet-data/lidc_crops"
SPLITS = ("train", "val", "test")

# (images, patients) as documented by the release, used to validate the download.
EXPECTED = {
    "train": (8843, 530),
    "val": (1993, 111),
    "test": (1980, 103),
}


def download(url, destination):
    """Stream a URL to disk, reporting progress on a single line."""

    with urllib.request.urlopen(url, timeout=60) as response:
        total = int(response.headers.get("Content-Length", 0))
        downloaded = 0
        with open(destination, "wb") as handle:
            while True:
                chunk = response.read(1 << 20)
                if not chunk:
                    break
                handle.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(
                        "\r  {:.1f}/{:.1f} MB ({:.0f}%)".format(
                            downloaded / 1e6, total / 1e6, 100 * downloaded / total
                        ),
                        end="",
                    )
        print()

    if total and downloaded != total:
        raise IOError("Truncated download: got {} of {} bytes".format(downloaded, total))


def extract(archive_path, split_dir):
    """Extract the archive so that ``split_dir`` directly contains images/ and gt/.

    The top-level directory name inside the archives is not documented, so it is
    detected rather than assumed.
    """

    with tempfile.TemporaryDirectory(dir=os.path.dirname(split_dir)) as staging:
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(staging)

        root = staging
        while True:
            entries = os.listdir(root)
            if "images" in entries and "gt" in entries:
                break
            subdirs = [e for e in entries if os.path.isdir(os.path.join(root, e))]
            if len(subdirs) != 1:
                raise IOError(
                    "Could not locate images/ and gt/ inside {} (found {})".format(
                        archive_path, entries
                    )
                )
            root = os.path.join(root, subdirs[0])

        if os.path.isdir(split_dir):
            shutil.rmtree(split_dir)
        shutil.move(root, split_dir)


def count_split(split_dir):
    """Return (number of images, number of patients) for an extracted split."""

    images_dir = os.path.join(split_dir, "images")
    patients = [p for p in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, p))]
    images = sum(
        len([f for f in os.listdir(os.path.join(images_dir, p)) if f.endswith(".png")])
        for p in patients
    )
    return images, len(patients)




def download_lidc(dest=DATA_ROOT, force=False, keep_archives=False, splits=SPLITS):
    dest = os.fspath(dest)
    os.makedirs(dest, exist_ok=True)
    ok = True
    for split in splits:
        split_dir = os.path.join(dest, split)
        archive_path = os.path.join(dest, "{}.tar.gz".format(split))
        if os.path.isdir(split_dir) and not force:
            print("{}: already extracted".format(split))
        else:
            if not os.path.isfile(archive_path) or force:
                print("{}: downloading".format(split))
                download("{}/{}.tar.gz".format(BASE_URL, split), archive_path)
            else:
                print("{}: archive already present".format(split))
            print("{}: extracting".format(split))
            extract(archive_path, split_dir)
            if not keep_archives:
                os.remove(archive_path)
        images, patients = count_split(split_dir)
        expected_images, expected_patients = EXPECTED[split]
        status = "OK" if (images, patients) == (expected_images, expected_patients) else "MISMATCH"
        ok = ok and status == "OK"
        print("{}: {} images, {} patients (expected {} / {}) [{}]".format(
            split, images, patients, expected_images, expected_patients, status))
    if not ok:
        raise RuntimeError("Counts differ from the published release; download may be incomplete.")
    print("Data ready under {}".format(os.path.abspath(dest)))

## Tensor Helpers

In [ ]:
def make_onehot(array, labels=None, axis=1, newaxis=False):
    if labels is None:
        labels = np.unique(array)
        labels = [label.item() for label in labels]

    new_shape = list(array.shape)
    if newaxis:
        new_shape.insert(axis, len(labels))
    else:
        new_shape[axis] *= len(labels)

    if torch.is_tensor(array):
        result = torch.zeros(new_shape, dtype=array.dtype, device=array.device)
    else:
        result = np.zeros(new_shape, dtype=array.dtype)

    n_seg_channels = 1 if newaxis else array.shape[axis]
    for seg_channel in range(n_seg_channels):
        for label_index, label in enumerate(labels):
            src_slice = [slice(None)] * len(array.shape)
            dst_slice = [slice(None)] * len(new_shape)
            dst_slice[axis] = seg_channel * len(labels) + label_index
            if not newaxis:
                src_slice[axis] = seg_channel
            result[tuple(dst_slice)] = array[tuple(src_slice)] == label
    return result


def match_to(x, ref, keep_axes=(1,)):
    if isinstance(keep_axes, int):
        keep_axes = (keep_axes,)
    target_shape = list(ref.shape)
    for axis in keep_axes:
        target_shape[axis] = x.shape[axis]
    if x.dim() == 1:
        x = x.unsqueeze(0)
    if x.dim() == 2:
        while x.dim() < len(target_shape):
            x = x.unsqueeze(-1)
    return x.expand(*target_shape).to(device=ref.device, dtype=ref.dtype)


make_onehot_segmentation = make_onehot

## Disagreement Targets

In [ ]:
def _binary_entropy(probability, eps=1e-6):
    probability = probability.clamp(eps, 1.0 - eps)
    return -(probability * torch.log(probability) + (1.0 - probability) * torch.log(1.0 - probability)) / math.log(2.0)


def compute_disagreement(masks, eps=1e-6):
    if torch.is_tensor(masks):
        return _binary_entropy(masks.float().mean(dim=1, keepdim=True), eps=eps)
    masks = np.asarray(masks, dtype=np.float32)
    probability = np.clip(masks.mean(axis=1, keepdims=True), eps, 1.0 - eps)
    entropy = -(probability * np.log(probability) + (1.0 - probability) * np.log(1.0 - probability))
    return entropy / np.log(2.0)


def model_uncertainty_from_samples(samples, foreground_channel=1, eps=1e-6, sample_dim=0):
    if sample_dim != 0:
        samples = samples.movedim(sample_dim, 0)
    if samples.shape[2] == 1:
        probabilities = torch.sigmoid(samples)
    else:
        probabilities = F.softmax(samples, dim=2)[:, :, foreground_channel:foreground_channel + 1]
    return _binary_entropy(probabilities.mean(dim=0), eps=eps)


def sample_diversity_from_samples(samples, foreground_channel=1, eps=1e-6, sample_dim=0):
    if sample_dim != 0:
        samples = samples.movedim(sample_dim, 0)
    if samples.shape[2] == 1:
        probabilities = torch.sigmoid(samples)
    else:
        probabilities = F.softmax(samples, dim=2)[:, :, foreground_channel:foreground_channel + 1]
    entropy_of_mean = _binary_entropy(probabilities.mean(dim=0), eps=eps)
    mean_entropy = _binary_entropy(probabilities, eps=eps).mean(dim=0)
    return (entropy_of_mean - mean_entropy).clamp(min=0.0)


def disagreement_alignment_loss(model_uncertainty, human_disagreement):
    return F.l1_loss(model_uncertainty, human_disagreement.float(), reduction="sum")

## Segmentation And Disagreement Metrics

In [ ]:
def assert_shape(test, reference):
    if test.shape != reference.shape:
        raise AssertionError("Shape mismatch: {} and {}".format(test.shape, reference.shape))


def disagreement_mae(model_uncertainty, human_disagreement):
    model_uncertainty = np.asarray(model_uncertainty, dtype=np.float32)
    human_disagreement = np.asarray(human_disagreement, dtype=np.float32)
    assert_shape(model_uncertainty, human_disagreement)
    return float(np.mean(np.abs(model_uncertainty - human_disagreement)))


def disagreement_correlation(model_uncertainty, human_disagreement, eps=1e-8):
    model_uncertainty = np.asarray(model_uncertainty, dtype=np.float32).ravel()
    human_disagreement = np.asarray(human_disagreement, dtype=np.float32).ravel()
    assert_shape(model_uncertainty, human_disagreement)
    model_uncertainty = model_uncertainty - model_uncertainty.mean()
    human_disagreement = human_disagreement - human_disagreement.mean()
    denom = np.sqrt(np.sum(model_uncertainty ** 2) * np.sum(human_disagreement ** 2))
    if denom < eps:
        return np.nan
    return float(np.sum(model_uncertainty * human_disagreement) / denom)


def binary_iou_distance(test, reference):
    test = np.asarray(test) != 0
    reference = np.asarray(reference) != 0
    assert_shape(test, reference)
    union = np.logical_or(test, reference).sum()
    if union == 0:
        return 0.0
    return float(1.0 - np.logical_and(test, reference).sum() / union)


def generalized_energy_distance(samples, references, distance=binary_iou_distance):
    samples = np.asarray(samples)
    references = np.asarray(references)
    if samples.shape[1:] != references.shape[1:]:
        raise AssertionError("Shape mismatch: {} and {}".format(samples.shape[1:], references.shape[1:]))
    sample_reference = np.mean([distance(sample, reference) for sample in samples for reference in references])
    sample_sample = np.mean([distance(a, b) for a in samples for b in samples])
    reference_reference = np.mean([distance(a, b) for a in references for b in references])
    return float(2.0 * sample_reference - sample_sample - reference_reference)


def dice(test, reference, nan_for_nonexisting=True):
    test = np.asarray(test) != 0
    reference = np.asarray(reference) != 0
    assert_shape(test, reference)
    tp = np.logical_and(test, reference).sum()
    fp = np.logical_and(test, ~reference).sum()
    fn = np.logical_and(~test, reference).sum()
    if not test.any() and not reference.any():
        return float("nan") if nan_for_nonexisting else 0.0
    return float(2.0 * tp / (2.0 * tp + fp + fn))


def jaccard(test, reference, nan_for_nonexisting=True):
    test = np.asarray(test) != 0
    reference = np.asarray(reference) != 0
    assert_shape(test, reference)
    union = np.logical_or(test, reference).sum()
    if union == 0:
        return float("nan") if nan_for_nonexisting else 0.0
    return float(np.logical_and(test, reference).sum() / union)

## Dataset

In [ ]:
NUM_GRADERS = 4
SPLITS = ("train", "val", "test")


def _load_png(path):
    with Image.open(path) as handle:
        return np.asarray(handle.convert("L"))


class LIDCCrops(Dataset):
    def __init__(self, root="data/lidc", split="train", crop_size=128, train=None, single_random_grader=False):
        if split not in SPLITS:
            raise ValueError("split must be one of {}, got {!r}".format(SPLITS, split))
        self.root = os.fspath(root)
        self.split = split
        self.crop_size = crop_size
        self.train = (split == "train") if train is None else train
        self.single_random_grader = single_random_grader
        self.images_dir = os.path.join(self.root, split, "images")
        self.gt_dir = os.path.join(self.root, split, "gt")
        if not os.path.isdir(self.images_dir):
            raise FileNotFoundError("No LIDC data at {}".format(os.path.abspath(os.path.join(self.root, split))))
        self.samples = self._build_index()
        if not self.samples:
            raise RuntimeError("Found no usable samples under {}".format(self.images_dir))

    def _build_index(self):
        samples = []
        incomplete = 0
        for patient in sorted(os.listdir(self.images_dir)):
            image_dir = os.path.join(self.images_dir, patient)
            if not os.path.isdir(image_dir):
                continue
            gt_dir = os.path.join(self.gt_dir, patient)
            for filename in sorted(os.listdir(image_dir)):
                if not filename.endswith(".png"):
                    continue
                stem = filename[:-4]
                mask_paths = [os.path.join(gt_dir, "{}_l{}.png".format(stem, grader)) for grader in range(NUM_GRADERS)]
                if all(os.path.isfile(path) for path in mask_paths):
                    samples.append((os.path.join(image_dir, filename), mask_paths))
                else:
                    incomplete += 1
        if incomplete:
            print("LIDCCrops[{}]: skipped {} crops without all grader masks".format(self.split, incomplete))
        return samples

    def _crop_origin(self, height, width, size):
        if not self.train:
            return (height - size) // 2, (width - size) // 2
        return (
            int(torch.randint(0, height - size + 1, (1,)).item()),
            int(torch.randint(0, width - size + 1, (1,)).item()),
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, mask_paths = self.samples[index]
        image = _load_png(image_path).astype(np.float32) / 255.0
        masks = np.stack([(_load_png(path) > 0).astype(np.float32) for path in mask_paths])
        if self.crop_size is not None:
            top, left = self._crop_origin(*image.shape, self.crop_size)
            image = image[top:top + self.crop_size, left:left + self.crop_size]
            masks = masks[:, top:top + self.crop_size, left:left + self.crop_size]
        sample = {
            "image": torch.from_numpy(np.ascontiguousarray(image))[None],
            "masks": torch.from_numpy(np.ascontiguousarray(masks)),
        }
        if self.single_random_grader:
            grader = int(torch.randint(0, NUM_GRADERS, (1,)).item())
            sample["target"] = sample["masks"][grader][None]
            sample["grader"] = grader
        return sample

## Probabilistic U-Net And Disagreement Head

In [ ]:
LOGVAR_MIN, LOGVAR_MAX = -10.0, 10.0


def is_conv(op):
    conv_types = (nn.Conv1d,
                  nn.Conv2d,
                  nn.Conv3d,
                  nn.ConvTranspose1d,
                  nn.ConvTranspose2d,
                  nn.ConvTranspose3d)
    if type(op) == type and issubclass(op, conv_types):
        return True
    elif type(op) in conv_types:
        return True
    else:
        return False



class ConvModule(nn.Module):

    def __init__(self, *args, **kwargs):

        super(ConvModule, self).__init__()

    def init_weights(self, init_fn, *args, **kwargs):

        class init_(object):

            def __init__(self):
                self.fn = init_fn
                self.args = args
                self.kwargs = kwargs

            def __call__(self, module):
                if is_conv(type(module)):
                    module.weight = self.fn(module.weight, *self.args, **self.kwargs)

        _init_ = init_()
        self.apply(_init_)

    def init_bias(self, init_fn, *args, **kwargs):

        class init_(object):

            def __init__(self):
                self.fn = init_fn
                self.args = args
                self.kwargs = kwargs

            def __call__(self, module):
                if is_conv(type(module)) and module.bias is not None:
                    module.bias = self.fn(module.bias, *self.args, **self.kwargs)

        _init_ = init_()
        self.apply(_init_)



class ConcatCoords(nn.Module):

    def forward(self, input_):

        dim = input_.dim() - 2
        coord_channels = []
        for i in range(dim):
            view = [1, ] * dim
            view[i] = -1
            repeat = list(input_.shape[2:])
            repeat[i] = 1
            coord_channels.append(
                torch.linspace(-0.5, 0.5, input_.shape[i+2])
                .view(*view)
                .repeat(*repeat)
                .to(device=input_.device, dtype=input_.dtype))
        coord_channels = torch.stack(coord_channels).unsqueeze(0)
        repeat = [1, ] * input_.dim()
        repeat[0] = input_.shape[0]
        coord_channels = coord_channels.repeat(*repeat).contiguous()

        return torch.cat([input_, coord_channels], 1)



class InjectionConvEncoder(ConvModule):

    _default_activation_kwargs = dict(inplace=True)
    _default_norm_kwargs = dict()
    _default_conv_kwargs = dict(kernel_size=3, padding=1)
    _default_pool_kwargs = dict(kernel_size=2)
    _default_dropout_kwargs = dict()
    _default_global_pool_kwargs = dict()

    def __init__(self,
                 in_channels=1,
                 out_channels=6,
                 depth=4,
                 injection_depth="last",
                 injection_channels=0,
                 block_depth=2,
                 num_feature_maps=24,
                 feature_map_multiplier=2,
                 activation_op=nn.LeakyReLU,
                 activation_kwargs=None,
                 norm_op=nn.InstanceNorm2d,
                 norm_kwargs=None,
                 norm_depth=0,
                 conv_op=nn.Conv2d,
                 conv_kwargs=None,
                 pool_op=nn.AvgPool2d,
                 pool_kwargs=None,
                 dropout_op=None,
                 dropout_kwargs=None,
                 global_pool_op=nn.AdaptiveAvgPool2d,
                 global_pool_kwargs=None,
                 **kwargs):

        super(InjectionConvEncoder, self).__init__(**kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.depth = depth
        self.injection_depth = depth - 1 if injection_depth == "last" else injection_depth
        self.injection_channels = injection_channels
        self.block_depth = block_depth
        self.num_feature_maps = num_feature_maps
        self.feature_map_multiplier = feature_map_multiplier

        self.activation_op = activation_op
        self.activation_kwargs = self._default_activation_kwargs
        if activation_kwargs is not None:
            self.activation_kwargs.update(activation_kwargs)

        self.norm_op = norm_op
        self.norm_kwargs = self._default_norm_kwargs
        if norm_kwargs is not None:
            self.norm_kwargs.update(norm_kwargs)
        self.norm_depth = depth if norm_depth == "full" else norm_depth

        self.conv_op = conv_op
        self.conv_kwargs = self._default_conv_kwargs
        if conv_kwargs is not None:
            self.conv_kwargs.update(conv_kwargs)

        self.pool_op = pool_op
        self.pool_kwargs = self._default_pool_kwargs
        if pool_kwargs is not None:
            self.pool_kwargs.update(pool_kwargs)

        self.dropout_op = dropout_op
        self.dropout_kwargs = self._default_dropout_kwargs
        if dropout_kwargs is not None:
            self.dropout_kwargs.update(dropout_kwargs)

        self.global_pool_op = global_pool_op
        self.global_pool_kwargs = self._default_global_pool_kwargs
        if global_pool_kwargs is not None:
            self.global_pool_kwargs.update(global_pool_kwargs)

        for d in range(self.depth):

            in_ = self.in_channels if d == 0 else self.num_feature_maps * (self.feature_map_multiplier**(d-1))
            out_ = self.num_feature_maps * (self.feature_map_multiplier**d)

            if d == self.injection_depth + 1:
                in_ += self.injection_channels

            layers = []
            if d > 0:
                layers.append(self.pool_op(**self.pool_kwargs))
            for b in range(self.block_depth):
                current_in = in_ if b == 0 else out_
                layers.append(self.conv_op(current_in, out_, **self.conv_kwargs))
                if self.norm_op is not None and d < self.norm_depth:
                    layers.append(self.norm_op(out_, **self.norm_kwargs))
                if self.activation_op is not None:
                    layers.append(self.activation_op(**self.activation_kwargs))
                if self.dropout_op is not None:
                    layers.append(self.dropout_op(**self.dropout_kwargs))
            if d == self.depth - 1:
                current_conv_kwargs = self.conv_kwargs.copy()
                current_conv_kwargs["kernel_size"] = 1
                current_conv_kwargs["padding"] = 0
                current_conv_kwargs["bias"] = False
                layers.append(self.conv_op(out_, out_channels, **current_conv_kwargs))

            self.add_module("encode_{}".format(d), nn.Sequential(*layers))

        if self.global_pool_op is not None:
            self.add_module("global_pool", self.global_pool_op(1, **self.global_pool_kwargs))

    def forward(self, x, injection=None):

        for d in range(self.depth):
            x = self._modules["encode_{}".format(d)](x)
            if d == self.injection_depth and self.injection_channels > 0:
                injection = match_to(injection, x, self.injection_channels)
                x = torch.cat([x, injection], 1)
        if hasattr(self, "global_pool"):
            x = self.global_pool(x)

        return x


class InjectionUNet(ConvModule):

    def __init__(
        self,
        depth=5,
        in_channels=4,
        out_channels=4,
        kernel_size=3,
        dilation=1,
        num_feature_maps=24,
        block_depth=2,
        num_1x1_at_end=3,
        injection_channels=3,
        injection_at="end",
        activation_op=nn.LeakyReLU,
        activation_kwargs=None,
        pool_op=nn.AvgPool2d,
        pool_kwargs=dict(kernel_size=2),
        dropout_op=None,
        dropout_kwargs=None,
        norm_op=nn.InstanceNorm2d,
        norm_kwargs=None,
        conv_op=nn.Conv2d,
        conv_kwargs=None,
        upconv_op=nn.ConvTranspose2d,
        upconv_kwargs=None,
        output_activation_op=None,
        output_activation_kwargs=None,
        return_bottom=False,
        coords=False,
        coords_dim=2,
        **kwargs
    ):

        super(InjectionUNet, self).__init__(**kwargs)

        self.depth = depth
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.padding = (self.kernel_size + (self.kernel_size-1) * (self.dilation-1)) // 2
        self.num_feature_maps = num_feature_maps
        self.block_depth = block_depth
        self.num_1x1_at_end = num_1x1_at_end
        self.injection_channels = injection_channels
        self.injection_at = injection_at
        self.activation_op = activation_op
        self.activation_kwargs = {} if activation_kwargs is None else activation_kwargs
        self.pool_op = pool_op
        self.pool_kwargs = {} if pool_kwargs is None else pool_kwargs
        self.dropout_op = dropout_op
        self.dropout_kwargs = {} if dropout_kwargs is None else dropout_kwargs
        self.norm_op = norm_op
        self.norm_kwargs = {} if norm_kwargs is None else norm_kwargs
        self.conv_op = conv_op
        self.conv_kwargs = {} if conv_kwargs is None else conv_kwargs
        self.upconv_op = upconv_op
        self.upconv_kwargs = {} if upconv_kwargs is None else upconv_kwargs
        self.output_activation_op = output_activation_op
        self.output_activation_kwargs = {} if output_activation_kwargs is None else output_activation_kwargs
        self.return_bottom = return_bottom
        if not coords:
            self.coords = [[], []]
        elif coords is True:
            self.coords = [list(range(depth)), []]
        else:
            self.coords = coords
        self.coords_dim = coords_dim

        self.last_activations = None
        self.last_features = None

        # BUILD ENCODER
        for d in range(self.depth):

            block = []
            if d > 0:
                block.append(self.pool_op(**self.pool_kwargs))

            for i in range(self.block_depth):

                # bottom block fixed to have depth 1
                if d == self.depth - 1 and i > 0:
                    continue

                out_size = self.num_feature_maps * 2**d
                if d == 0 and i == 0:
                    in_size = self.in_channels
                elif i == 0:
                    in_size = self.num_feature_maps * 2**(d - 1)
                else:
                    in_size = out_size

                # check for coord appending at this depth
                if d in self.coords[0] and i == 0:
                    block.append(ConcatCoords())
                    in_size += self.coords_dim

                block.append(self.conv_op(in_size,
                                          out_size,
                                          self.kernel_size,
                                          padding=self.padding,
                                          dilation=self.dilation,
                                          **self.conv_kwargs))
                if self.dropout_op is not None:
                    block.append(self.dropout_op(**self.dropout_kwargs))
                if self.norm_op is not None:
                    block.append(self.norm_op(out_size, **self.norm_kwargs))
                block.append(self.activation_op(**self.activation_kwargs))

            self.add_module("encode-{}".format(d), nn.Sequential(*block))

        # BUILD DECODER
        for d in reversed(range(self.depth)):

            block = []

            for i in range(self.block_depth):

                # bottom block fixed to have depth 1
                if d == self.depth - 1 and i > 0:
                    continue

                out_size = self.num_feature_maps * 2**(d)
                if i == 0 and d < self.depth - 1:
                    in_size = self.num_feature_maps * 2**(d+1)
                elif i == 0 and self.injection_at == "bottom":
                    in_size = out_size + self.injection_channels
                else:
                    in_size = out_size

                # check for coord appending at this depth
                if d in self.coords[0] and i == 0 and d < self.depth - 1:
                    block.append(ConcatCoords())
                    in_size += self.coords_dim

                block.append(self.conv_op(in_size,
                                          out_size,
                                          self.kernel_size,
                                          padding=self.padding,
                                          dilation=self.dilation,
                                          **self.conv_kwargs))
                if self.dropout_op is not None:
                    block.append(self.dropout_op(**self.dropout_kwargs))
                if self.norm_op is not None:
                    block.append(self.norm_op(out_size, **self.norm_kwargs))
                block.append(self.activation_op(**self.activation_kwargs))

            if d > 0:
                block.append(self.upconv_op(out_size,
                                            out_size // 2,
                                            self.kernel_size,
                                            2,
                                            padding=self.padding,
                                            dilation=self.dilation,
                                            output_padding=1,
                                            **self.upconv_kwargs))

            self.add_module("decode-{}".format(d), nn.Sequential(*block))

        if self.injection_at == "end":
            out_size += self.injection_channels
        in_size = out_size
        for i in range(self.num_1x1_at_end):
            if i == self.num_1x1_at_end - 1:
                out_size = self.out_channels
            current_conv_kwargs = self.conv_kwargs.copy()
            current_conv_kwargs["bias"] = True
            self.add_module("reduce-{}".format(i), self.conv_op(in_size, out_size, 1, **current_conv_kwargs))
            if i != self.num_1x1_at_end - 1:
                self.add_module("reduce-{}-nonlin".format(i), self.activation_op(**self.activation_kwargs))
        if self.output_activation_op is not None:
            self.add_module("output-activation", self.output_activation_op(**self.output_activation_kwargs))

    def reset(self):

        self.last_activations = None
        self.last_features = None

    def forward(self, x, injection=None, reuse_last_activations=False, store_activations=False):

        if self.injection_at == "bottom":  # not worth it for now
            reuse_last_activations = False
            store_activations = False

        if self.last_activations is None or reuse_last_activations is False:

            enc = [x]

            for i in range(self.depth - 1):
                enc.append(self._modules["encode-{}".format(i)](enc[-1]))

            bottom_rep = self._modules["encode-{}".format(self.depth - 1)](enc[-1])

            if self.injection_at == "bottom" and self.injection_channels > 0:
                injection = match_to(injection, bottom_rep, (0, 1))
                bottom_rep = torch.cat((bottom_rep, injection), 1)

            x = self._modules["decode-{}".format(self.depth - 1)](bottom_rep)

            for i in reversed(range(self.depth - 1)):
                x = self._modules["decode-{}".format(i)](torch.cat((enc[-(self.depth - 1 - i)], x), 1))

            self.last_features = x
            if store_activations:
                self.last_activations = x.detach()

        else:

            x = self.last_activations

        if self.injection_at == "end" and self.injection_channels > 0:
            injection = match_to(injection, x, (0, 1))
            x = torch.cat((x, injection), 1)

        for i in range(self.num_1x1_at_end):
            x = self._modules["reduce-{}".format(i)](x)
            # Without these the 1x1 stack collapses to a single affine map. Since
            # the injection is spatially constant, that makes (out1 - out0) equal
            # S(x, y) + c(z), i.e. every sample a level set of one function -- so
            # all samples come out strictly nested and z can only dilate/erode the
            # mask globally. Measured at 100% nested on the LIDC test set before
            # this fix. See Kohl et al.'s fcomb.
            if i != self.num_1x1_at_end - 1:
                x = self._modules["reduce-{}-nonlin".format(i)](x)
        if self.output_activation_op is not None:
            x = self._modules["output-activation"](x)

        if self.return_bottom and not reuse_last_activations:
            return x, bottom_rep
        else:
            return x



class ProbabilisticSegmentationNet(ConvModule):

    def __init__(self,
                 in_channels=4,
                 out_channels=4,
                 num_feature_maps=24,
                 latent_size=3,
                 depth=5,
                 latent_distribution=torch.distributions.Normal,
                 task_op=InjectionUNet,
                 task_kwargs=None,
                 prior_op=InjectionConvEncoder,
                 prior_kwargs=None,
                 posterior_op=InjectionConvEncoder,
                 posterior_kwargs=None,
                 **kwargs):

        super(ProbabilisticSegmentationNet, self).__init__(**kwargs)

        self.task_op = task_op
        self.task_kwargs = {} if task_kwargs is None else task_kwargs
        self.prior_op = prior_op
        self.prior_kwargs = {} if prior_kwargs is None else prior_kwargs
        self.posterior_op = posterior_op
        self.posterior_kwargs = {} if posterior_kwargs is None else posterior_kwargs

        default_task_kwargs = dict(
            in_channels=in_channels,
            out_channels=out_channels,
            num_feature_maps=num_feature_maps,
            injection_size=latent_size,
            depth=depth
        )

        default_prior_kwargs = dict(
            in_channels=in_channels,
            num_feature_maps=num_feature_maps,
            z_dim=latent_size,
            depth=depth
        )

        default_posterior_kwargs = dict(
            in_channels=in_channels+out_channels,
            num_feature_maps=num_feature_maps,
            z_dim=latent_size,
            depth=depth
        )

        default_task_kwargs.update(self.task_kwargs)
        self.task_kwargs = default_task_kwargs
        default_prior_kwargs.update(self.prior_kwargs)
        self.prior_kwargs = default_prior_kwargs
        default_posterior_kwargs.update(self.posterior_kwargs)
        self.posterior_kwargs = default_posterior_kwargs

        self.latent_distribution = latent_distribution
        self._prior = None
        self._posterior = None

        self.make_modules()

    def make_modules(self):

        if type(self.task_op) == type:
            self.add_module("task_net", self.task_op(**self.task_kwargs))
        else:
            self.add_module("task_net", self.task_op)
        if type(self.prior_op) == type:
            self.add_module("prior_net", self.prior_op(**self.prior_kwargs))
        else:
            self.add_module("prior_net", self.prior_op)
        if type(self.posterior_op) == type:
            self.add_module("posterior_net", self.posterior_op(**self.posterior_kwargs))
        else:
            self.add_module("posterior_net", self.posterior_op)

    @property
    def prior(self):
        return self._prior

    @property
    def posterior(self):
        return self._posterior

    @property
    def last_activations(self):
        return self.task_net.last_activations

    def train(self, mode=True):

        super(ProbabilisticSegmentationNet, self).train(mode)
        self.reset()

    def reset(self):

        self.task_net.reset()
        self._prior = None
        self._posterior = None

    def forward(self, input_, seg=None, make_onehot=True, make_onehot_classes=None, newaxis=False):
        """Forward pass includes reparametrization sampling during training, otherwise it'll just take the prior mean."""

        self.encode_prior(input_)
        if self.training:
            self.encode_posterior(input_, seg, make_onehot, make_onehot_classes, newaxis)
            sample = self.posterior.rsample()
        else:
            sample = self.prior.loc
        return self.task_net(input_, sample, store_activations=not self.training)

    def encode_prior(self, input_):

        rep = self.prior_net(input_)
        if isinstance(rep, tuple):
            mean, logvar = rep
        elif torch.is_tensor(rep):
            mean, logvar = torch.split(rep, rep.shape[1] // 2, dim=1)
        # Clamped because the alignment loss backprops into the prior and the
        # cheapest way for it to raise sample diversity is to inflate the prior
        # variance; unclamped, logvar drifts up until exp() overflows and the
        # whole encoder goes NaN. exp(0.5*10) ~ 148 is far wider than anything a
        # 6-D latent needs, so this only ever catches the runaway.
        logvar = logvar.clamp(LOGVAR_MIN, LOGVAR_MAX)
        self._prior = self.latent_distribution(mean, logvar.mul(0.5).exp())
        return self._prior

    def encode_posterior(self, input_, seg, make_onehot=True, make_onehot_classes=None, newaxis=False):

        if make_onehot:
            if make_onehot_classes is None:
                make_onehot_classes = tuple(range(self.posterior_net.in_channels - input_.shape[1]))
            seg = make_onehot_segmentation(seg, make_onehot_classes, newaxis=newaxis)
        rep = self.posterior_net(torch.cat((input_, seg.float()), 1))
        if isinstance(rep, tuple):
            mean, logvar = rep
        elif torch.is_tensor(rep):
            mean, logvar = torch.split(rep, rep.shape[1] // 2, dim=1)
        logvar = logvar.clamp(LOGVAR_MIN, LOGVAR_MAX)
        self._posterior = self.latent_distribution(mean, logvar.mul(0.5).exp())
        return self._posterior

    def sample_prior(self, N=1, out_device=None, input_=None):
        """Draw multiple samples from the current prior.
        
        * input_ is required if no activations are stored in task_net.
        * If input_ is given, prior will automatically be encoded again.
        * Returns either a single sample or a list of samples.

        """

        if out_device is None:
            if self.last_activations is not None:
                out_device = self.last_activations.device
            elif input_ is not None:
                out_device = input_.device
            else:
                out_device = next(self.task_net.parameters()).device
        with torch.no_grad():
            if self.prior is None or input_ is not None:
                self.encode_prior(input_)
            result = []
            if input_ is not None:
                result.append(self.task_net(input_, self.prior.sample(), reuse_last_activations=False, store_activations=True).to(device=out_device))
            while len(result) < N:
                result.append(self.task_net(input_,
                                            self.prior.sample(),
                                            reuse_last_activations=self.last_activations is not None,
                                            store_activations=False).to(device=out_device))
            if N == 1:
                return result[0]
            else:
                return result

    def reconstruct(self, sample=None, use_posterior_mean=True, out_device=None, input_=None):
        """Reconstruct a sample or the current posterior mean. Will not compute gradients!"""

        if self.posterior is None and sample is None:
            raise ValueError("'posterior' is currently None. Please pass an input and a segmentation first.")
        if out_device is None:
            out_device = next(self.task_net.parameters()).device
        if sample is None:
            if use_posterior_mean:
                sample = self.posterior.loc
            else:
                sample = self.posterior.sample()
        else:
            sample = sample.to(next(self.task_net.parameters()).device)
        with torch.no_grad():
            return self.task_net(input_, sample, reuse_last_activations=True).to(device=out_device)

    def kl_divergence(self):
        """Compute current KL, requires existing prior and posterior."""

        if self.posterior is None or self.prior is None:
            raise ValueError("'prior' and 'posterior' must not be None, but prior={} and posterior={}".format(self.prior, self.posterior))
        return torch.distributions.kl_divergence(self.posterior, self.prior).sum()

    def elbo(self, seg, input_=None, nll_reduction="sum", beta=1.0, make_onehot=True, make_onehot_classes=None, newaxis=False):
        """Compute the ELBO with seg as ground truth.

        * Prior is expected and will not be encoded.
        * If input_ is given, posterior will automatically be encoded.
        * Either input_ or stored activations must be available.

        """

        if self.last_activations is None:
            raise ValueError("'last_activations' is currently None. Please pass an input first.")
        if input_ is not None:
            with torch.no_grad():
                self.encode_posterior(input_, seg, make_onehot=make_onehot, make_onehot_classes=make_onehot_classes, newaxis=newaxis)
        if make_onehot and newaxis:
            pass  # seg will already be (B x SPACE)
        elif make_onehot and not newaxis:
            seg = seg[:, 0]  # in this case seg will hopefully be (B x 1 x SPACE)
        else:
            seg = torch.argmax(seg, 1, keepdim=False)  # seg is already onehot
        kl = self.kl_divergence()
        nll = nn.NLLLoss(reduction=nll_reduction)(self.reconstruct(sample=None, use_posterior_mean=True, out_device=None), seg.long())
        return - (beta * nll + kl)


class DisagreementAwareProbabilisticSegmentationNet(ProbabilisticSegmentationNet):
    """Probabilistic U-Net with explicit human-disagreement supervision."""

    def __init__(self,
                 *args,
                 disagreement_channels=1,
                 foreground_channel=1,
                 **kwargs):

        self.disagreement_channels = disagreement_channels
        self.foreground_channel = foreground_channel
        super(DisagreementAwareProbabilisticSegmentationNet, self).__init__(*args, **kwargs)

    def make_modules(self):

        super(DisagreementAwareProbabilisticSegmentationNet, self).make_modules()
        feature_channels = self.task_net.num_feature_maps
        conv_op = self.task_net.conv_op
        self.add_module(
            "disagreement_head",
            conv_op(feature_channels, self.disagreement_channels, kernel_size=1)
        )

    def predict_disagreement(self):
        """Predict human disagreement from the latest U-Net spatial features."""

        if self.task_net.last_features is None:
            raise ValueError("No stored U-Net features. Run a forward pass before predicting disagreement.")
        return torch.sigmoid(self.disagreement_head(self.task_net.last_features))

    def sample_prior_train(self, input_, n_samples=4):
        """Draw differentiable prior samples and decode them for alignment loss.

        The prior's parameters are detached on purpose. The alignment loss should
        shape how the decoder turns z into a segmentation; if it can also reach
        the prior, the cheapest way for it to raise sample diversity is to inflate
        the prior variance without bound, which makes KL double every step until
        it overflows. Detaching leaves the prior to the KL term, as in Kohl et al.
        """

        if self.prior is None:
            self.encode_prior(input_)

        prior = self.latent_distribution(
            self.prior.loc.detach(), self.prior.scale.detach()
        )

        outputs = []
        for _ in range(n_samples):
            outputs.append(self.task_net(input_, prior.rsample(), reuse_last_activations=False))
        return torch.stack(outputs, dim=0)

    def disagreement_losses(self,
                            input_,
                            masks,
                            n_samples=4,
                            lambda_disagreement=1.0,
                            lambda_alignment=0.0,
                            eps=1e-6):
        """Compute auxiliary disagreement-prediction and diversity-alignment losses.

        Args:
            input_: Image tensor used to encode the prior for sample diversity.
            masks: Multi-rater binary masks with shape ``(B, G, *spatial)``.
            n_samples: Number of prior samples used for model uncertainty.
            lambda_disagreement: Weight for MSE disagreement supervision.
            lambda_alignment: Weight for uncertainty/disagreement alignment.
            eps: Numerical stability constant.

        Returns:
            ``(loss, metrics)`` where metrics contains detached scalar terms and
            maps useful for logging/visualization.
        """

        human_disagreement = compute_disagreement(masks, eps=eps)
        predicted_disagreement = self.predict_disagreement()
        # reduction="sum" to match the summed reconstruction NLL. As a mean this is
        # ~B*H*W = 5e5 times smaller than loss_seg, so lambda_disagreement * L_D was
        # ~1e-6 of the total and the head received no gradient at all.
        loss_disagreement = F.mse_loss(
            predicted_disagreement, human_disagreement.float(), reduction="sum"
        )

        loss_alignment = predicted_disagreement.new_tensor(0.0)
        model_uncertainty = None
        if lambda_alignment != 0.0:
            samples = self.sample_prior_train(input_, n_samples=n_samples)
            model_uncertainty = model_uncertainty_from_samples(
                samples,
                foreground_channel=self.foreground_channel,
                eps=eps
            )
            loss_alignment = disagreement_alignment_loss(model_uncertainty, human_disagreement)

        loss = lambda_disagreement * loss_disagreement + lambda_alignment * loss_alignment
        metrics = {
            "loss_disagreement": loss_disagreement.detach(),
            "loss_alignment": loss_alignment.detach(),
            "human_disagreement": human_disagreement.detach(),
            "predicted_disagreement": predicted_disagreement.detach(),
        }
        if model_uncertainty is not None:
            metrics["model_uncertainty"] = model_uncertainty.detach()
        return loss, metrics


## Training

In [ ]:
VARIANTS = ("baseline", "head", "full")


def limit_dataset(dataset, max_items):
    if max_items is None or max_items >= len(dataset):
        return dataset
    return Subset(dataset, range(max_items))


def make_loaders(args):
    train_set = LIDCCrops(
        root=args.data_root,
        split="train",
        crop_size=args.crop_size,
        single_random_grader=True,
    )
    val_set = LIDCCrops(
        root=args.data_root,
        split=args.eval_split,
        crop_size=args.crop_size,
        train=False,
        single_random_grader=True,
    )
    train_set = limit_dataset(train_set, args.max_train)
    val_set = limit_dataset(val_set, args.max_val)

    train_loader = DataLoader(
        train_set,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=args.device.startswith("cuda"),
    )
    val_loader = DataLoader(
        val_set,
        batch_size=args.eval_batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=args.device.startswith("cuda"),
    )
    return train_loader, val_loader


def make_model(args):
    model_cls = (
        ProbabilisticSegmentationNet
        if args.variant == "baseline"
        else DisagreementAwareProbabilisticSegmentationNet
    )
    return model_cls(
        in_channels=1,
        out_channels=2,
        num_feature_maps=args.feature_maps,
        latent_size=args.latent_size,
        depth=args.depth,
        latent_distribution=distributions.Normal,
        task_op=InjectionUNet,
        task_kwargs={
            "injection_channels": args.latent_size,
            "output_activation_op": nn.LogSoftmax,
            "output_activation_kwargs": {"dim": 1},
            "activation_kwargs": {"inplace": True},
        },
        prior_op=InjectionConvEncoder,
        prior_kwargs={
            "in_channels": 1,
            "out_channels": 2 * args.latent_size,
            "depth": args.depth,
            "num_feature_maps": args.feature_maps,
            "activation_kwargs": {"inplace": True},
            "norm_depth": 0,
        },
        posterior_op=InjectionConvEncoder,
        posterior_kwargs={
            "in_channels": 3,
            "out_channels": 2 * args.latent_size,
            "depth": args.depth,
            "num_feature_maps": args.feature_maps,
            "activation_kwargs": {"inplace": True},
            "norm_depth": 0,
        },
    ).to(args.device)


def kl_loss(model):
    kl = distributions.kl_divergence(model.posterior, model.prior)
    return kl.reshape(kl.shape[0], -1).sum(dim=1).mean()


















def set_lr(optimizer, step, args):
    if args.lr_final >= args.lr:
        return args.lr
    decay_every = max(1, args.steps // args.lr_decay_steps)
    decay_index = min((step - 1) // decay_every, args.lr_decay_steps)
    gamma = (args.lr_final / args.lr) ** (1.0 / args.lr_decay_steps)
    lr = args.lr * (gamma ** decay_index)
    for group in optimizer.param_groups:
        group["lr"] = lr
    return lr


def batch_to_device(batch, device):
    return {
        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def train_step(model, batch, optimizer, criterion, args, step):
    model.train()
    model.reset()
    optimizer.zero_grad()
    lr = set_lr(optimizer, step, args)

    image = batch["image"]
    target = batch["target"].long()
    masks = batch["masks"]

    prediction = model(image, target, make_onehot=True, make_onehot_classes=(0, 1))
    loss_seg = criterion(prediction, target[:, 0].long())
    loss_kl = kl_loss(model)
    loss = loss_seg + args.beta * loss_kl
    log = {
        "step": step,
        "lr": lr,
        "loss": loss.detach(),
        "loss_seg": loss_seg.detach(),
        "loss_kl": loss_kl.detach(),
        "loss_disagreement": prediction.new_tensor(0.0),
        "loss_alignment": prediction.new_tensor(0.0),
    }

    if args.variant in ("head", "full"):
        lambda_alignment = args.lambda_alignment if args.variant == "full" else 0.0
        loss_aux, aux_metrics = model.disagreement_losses(
            image,
            masks,
            n_samples=args.train_samples,
            lambda_disagreement=args.lambda_disagreement,
            lambda_alignment=lambda_alignment,
        )
        loss = loss + loss_aux
        log["loss"] = loss.detach()
        log["loss_disagreement"] = aux_metrics["loss_disagreement"]
        log["loss_alignment"] = aux_metrics["loss_alignment"]

    loss.backward()
    if args.grad_clip > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
    optimizer.step()
    return {key: float(value.detach().cpu()) if torch.is_tensor(value) else value for key, value in log.items()}

def sample_masks(model, image, n_samples):
    outputs = model.sample_prior(n_samples, out_device=image.device, input_=image)
    outputs = torch.stack(outputs, dim=0)
    masks = torch.argmax(outputs, dim=2)
    return outputs, masks


def evaluate(model, loader, args):
    model.eval()
    dice_scores = []
    iou_scores = []
    ged_scores = []
    disagreement_maes = []
    disagreement_corrs = []
    predicted_maes = []
    predicted_corrs = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False, dynamic_ncols=True):
            batch = batch_to_device(batch, args.device)
            image = batch["image"]
            masks = batch["masks"]

            model.reset()
            prediction = model(image)
            pred_mask = torch.argmax(prediction, dim=1).cpu().numpy()
            grader_masks = masks.cpu().numpy().astype(bool)

            outputs, prior_masks = sample_masks(model, image, args.eval_samples)
            model_uncertainty = model_uncertainty_from_samples(outputs).cpu().numpy()
            human_disagreement = compute_disagreement(masks).cpu().numpy()

            predicted_disagreement = None
            if args.variant in ("head", "full"):
                predicted_disagreement = model.predict_disagreement().cpu().numpy()

            prior_masks = prior_masks.cpu().numpy().astype(bool)
            for i in range(image.shape[0]):
                for grader in range(grader_masks.shape[1]):
                    dice_scores.append(
                        dice(pred_mask[i] != 0, grader_masks[i, grader], nan_for_nonexisting=True)
                    )
                    iou_scores.append(
                        jaccard(pred_mask[i] != 0, grader_masks[i, grader], nan_for_nonexisting=True)
                    )
                ged_scores.append(generalized_energy_distance(prior_masks[:, i], grader_masks[i]))
                disagreement_maes.append(
                    disagreement_mae(model_uncertainty[i, 0], human_disagreement[i, 0])
                )
                disagreement_corrs.append(
                    disagreement_correlation(model_uncertainty[i, 0], human_disagreement[i, 0])
                )
                if predicted_disagreement is not None:
                    predicted_maes.append(
                        disagreement_mae(predicted_disagreement[i, 0], human_disagreement[i, 0])
                    )
                    predicted_corrs.append(
                        disagreement_correlation(predicted_disagreement[i, 0], human_disagreement[i, 0])
                    )

    result = {
        "val_dice": float(np.nanmean(dice_scores)),
        "val_iou": float(np.nanmean(iou_scores)),
        "val_ged": float(np.nanmean(ged_scores)),
        "val_uncertainty_mae": float(np.nanmean(disagreement_maes)),
        "val_uncertainty_corr": float(np.nanmean(disagreement_corrs)),
    }
    if predicted_maes:
        result["val_predicted_disagreement_mae"] = float(np.nanmean(predicted_maes))
        result["val_predicted_disagreement_corr"] = float(np.nanmean(predicted_corrs))
    return result


def append_csv(path, row):
    exists = os.path.exists(path)
    with open(path, "a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=sorted(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def save_checkpoint(path, model, optimizer, step, args, metrics):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "step": step,
            "args": vars(args),
            "metrics": metrics,
            "rng_state": {
                "python": random.getstate(),
                "numpy": np.random.get_state(),
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            },
        },
        path,
    )




def best_ged_from_history(path):
    if not os.path.exists(path):
        return None
    values = []
    with open(path, newline="") as handle:
        for row in csv.DictReader(handle):
            if row.get("val_ged") not in (None, ""):
                values.append(float(row["val_ged"]))
    if not values:
        return None
    return min(values)


## Held-Out Test Evaluation

In [ ]:
VARIANTS = ("baseline", "head", "full")


def load_checkpoint_model(path, device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    saved = dict(checkpoint["args"])
    saved["device"] = device
    model = make_model(SimpleNamespace(**saved))
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint, saved["variant"]


def boundary_band(mask, kernel=5):
    """Morphological gradient of a binary mask: dilation minus erosion.

    Implemented with max-pooling because scipy is not installed in this env.
    ``mask`` is a float tensor shaped (B, 1, H, W).
    """

    dilated = F.max_pool2d(mask, kernel, stride=1, padding=kernel // 2)
    eroded = -F.max_pool2d(-mask, kernel, stride=1, padding=kernel // 2)
    return dilated - eroded


def nested_and_diversity(sample_masks_np):
    """Return (mean pairwise 1-IoU, whether all samples are nested) for one image.

    ``sample_masks_np`` is a boolean array shaped (K, H, W).
    """

    k = sample_masks_np.shape[0]
    distances = [
        binary_iou_distance(sample_masks_np[i], sample_masks_np[j])
        for i in range(k)
        for j in range(k)
        if i != j
    ]
    order = np.argsort(sample_masks_np.reshape(k, -1).sum(axis=1))
    nested = all(
        not np.logical_and(sample_masks_np[order[i]], ~sample_masks_np[order[i + 1]]).any()
        for i in range(k - 1)
    )
    return float(np.mean(distances)) if distances else 0.0, bool(nested)


def evaluate_checkpoint(model, variant, loader, args):
    """Test-set metrics for one checkpoint. Mirrors train_lidc_ablation.evaluate,
    with the diversity diagnostics and the trivial control added."""

    acc = {key: [] for key in (
        "dice", "iou", "ged",
        "unc_mae", "unc_corr", "div_mae", "div_corr",
        "pairwise_distance", "nested",
        "pred_mae", "pred_corr", "control_corr",
    )}

    with torch.no_grad():
        for batch in tqdm(loader, desc=variant, leave=False, dynamic_ncols=True):
            image = batch["image"].to(args.device)
            masks = batch["masks"].to(args.device)

            model.reset()
            prediction = model(image)
            pred_mask = torch.argmax(prediction, dim=1).cpu().numpy()

            outputs, prior_masks = sample_masks(model, image, args.eval_samples)
            uncertainty = model_uncertainty_from_samples(outputs).cpu().numpy()
            diversity = sample_diversity_from_samples(outputs).cpu().numpy()
            human = compute_disagreement(masks).cpu().numpy()

            # Trivial Q1 control: outline the model's own deterministic prediction.
            deterministic = torch.argmax(prediction, dim=1, keepdim=True).float()
            control = boundary_band(deterministic).cpu().numpy()

            predicted = None
            if variant in ("head", "full"):
                predicted = model.predict_disagreement().cpu().numpy()

            grader_masks = masks.cpu().numpy().astype(bool)
            prior_masks = prior_masks.cpu().numpy().astype(bool)

            for i in range(image.shape[0]):
                for grader in range(grader_masks.shape[1]):
                    acc["dice"].append(
                        dice(pred_mask[i] != 0, grader_masks[i, grader], nan_for_nonexisting=True)
                    )
                    acc["iou"].append(
                        jaccard(pred_mask[i] != 0, grader_masks[i, grader], nan_for_nonexisting=True)
                    )
                acc["ged"].append(generalized_energy_distance(prior_masks[:, i], grader_masks[i]))

                acc["unc_mae"].append(disagreement_mae(uncertainty[i, 0], human[i, 0]))
                acc["unc_corr"].append(disagreement_correlation(uncertainty[i, 0], human[i, 0]))
                acc["div_mae"].append(disagreement_mae(diversity[i, 0], human[i, 0]))
                acc["div_corr"].append(disagreement_correlation(diversity[i, 0], human[i, 0]))
                acc["control_corr"].append(disagreement_correlation(control[i, 0], human[i, 0]))

                distance, nested = nested_and_diversity(prior_masks[:, i])
                acc["pairwise_distance"].append(distance)
                acc["nested"].append(float(nested))

                if predicted is not None:
                    acc["pred_mae"].append(disagreement_mae(predicted[i, 0], human[i, 0]))
                    acc["pred_corr"].append(disagreement_correlation(predicted[i, 0], human[i, 0]))

    return {
        key: (float(np.nanmean(values)) if values else float("nan"))
        for key, values in acc.items()
    }

## Qualitative Grid Figures

In [ ]:
def torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def checkpoint_args(checkpoint, device, variant):
    saved = dict(checkpoint.get("args") or {})
    saved["device"] = device
    saved["variant"] = variant
    return SimpleNamespace(**saved)


def load_model(checkpoint_path, variant, device):
    checkpoint = torch_load(checkpoint_path, device)
    args = checkpoint_args(checkpoint, device, variant)
    model = make_model(args)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint


def sample_model(model, image, n_samples, has_disagreement_head):
    with torch.no_grad():
        model.reset()
        outputs = model.sample_prior(n_samples, out_device=image.device, input_=image)
        outputs = torch.stack(outputs, dim=0)
        masks = torch.argmax(outputs, dim=2).cpu().numpy().astype(np.float32)
        uncertainty = model_uncertainty_from_samples(outputs).cpu().numpy()
        predicted_disagreement = None
        if has_disagreement_head:
            predicted_disagreement = model.predict_disagreement().cpu().numpy()
    return masks[:, 0], uncertainty[0, 0], (
        None if predicted_disagreement is None else predicted_disagreement[0, 0]
    )


def disagreement_score(sample):
    disagreement = compute_disagreement(sample["masks"][None]).numpy()[0, 0]
    return float(disagreement.mean())


def case_identity(dataset, index):
    image_path, _ = dataset.samples[index]
    patient = os.path.basename(os.path.dirname(image_path))
    stem = os.path.splitext(os.path.basename(image_path))[0]
    return patient, stem


def load_case(dataset, index):
    patient, stem = case_identity(dataset, index)
    return {
        "index": index,
        "patient": patient,
        "stem": stem,
        "sample": dataset[index],
    }


def choose_cases(dataset, requested_indices, num_cases, selection):
    if requested_indices:
        return [load_case(dataset, index) for index in requested_indices[:num_cases]]
    if selection == "first":
        return [load_case(dataset, index) for index in range(min(num_cases, len(dataset)))]

    scores = []
    for index in tqdm(range(len(dataset)), desc="rank disagreement", dynamic_ncols=True):
        case = load_case(dataset, index)
        scores.append((disagreement_score(case["sample"]), case))
    # key= on the score alone: cases are dicts, so a tie would otherwise try to
    # compare them and raise.
    scores.sort(key=lambda item: item[0], reverse=True)

    # LIDC crops are consecutive slices through the same nodule, so the top of this
    # ranking is one high-disagreement lesion repeated dozens of times -- without
    # this, "8 cases" is really one case shown 8 times. Take one crop per patient
    # first, and only reuse patients if that does not fill num_cases.
    chosen, seen, leftovers = [], set(), []
    for _, case in scores:
        if case["patient"] in seen:
            leftovers.append(case)
            continue
        seen.add(case["patient"])
        chosen.append(case)
        if len(chosen) == num_cases:
            return chosen
    return (chosen + leftovers)[:num_cases]


def add_panel(ax, image, title, cmap="gray", vmin=None, vmax=None):
    ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def make_case_figure(
    out_path,
    title,
    image,
    masks,
    human_disagreement,
    baseline_samples,
    baseline_uncertainty,
    full_samples,
    full_uncertainty,
    predicted_disagreement,
):
    n_samples = baseline_samples.shape[0]
    ncols = max(6, n_samples + 2)
    fig, axes = plt.subplots(4, ncols, figsize=(2.1 * ncols, 8.2))
    fig.suptitle(title, fontsize=10)

    for ax in axes.ravel():
        ax.axis("off")

    add_panel(axes[0, 0], image, "input")
    for grader in range(masks.shape[0]):
        add_panel(axes[0, grader + 1], masks[grader], "human M{}".format(grader + 1))
    add_panel(axes[0, 5], human_disagreement, "human D", cmap="magma", vmin=0.0, vmax=1.0)

    axes[1, 0].set_title("baseline samples", fontsize=8)
    axes[1, 0].axis("off")
    for i in range(n_samples):
        add_panel(axes[1, i + 1], baseline_samples[i], "S{}".format(i + 1))
    add_panel(
        axes[1, n_samples + 1],
        baseline_uncertainty,
        "baseline U",
        cmap="magma",
        vmin=0.0,
        vmax=1.0,
    )

    axes[2, 0].set_title("full samples", fontsize=8)
    axes[2, 0].axis("off")
    for i in range(n_samples):
        add_panel(axes[2, i + 1], full_samples[i], "S{}".format(i + 1))
    add_panel(
        axes[2, n_samples + 1],
        full_uncertainty,
        "full U",
        cmap="magma",
        vmin=0.0,
        vmax=1.0,
    )

    add_panel(
        axes[3, 0],
        predicted_disagreement,
        "predicted D",
        cmap="magma",
        vmin=0.0,
        vmax=1.0,
    )
    add_panel(axes[3, 1], np.abs(full_uncertainty - human_disagreement), "|U-D|", cmap="magma")
    add_panel(
        axes[3, 2],
        np.abs(predicted_disagreement - human_disagreement),
        "|Dhat-D|",
        cmap="magma",
    )

    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(out_path, dpi=180)
    plt.close(fig)

## Architecture Panels

In [ ]:
SIZE_PX = 296
CMAP = "inferno"


def lesion_window(masks, margin=1.8, min_size=24):
    """Square window centred on the union of the grader masks.

    Returns (top, left, size) in pixels, clipped to the image.
    """

    union = masks.max(axis=0) > 0.5
    height, width = union.shape
    if not union.any():
        return 0, 0, min(height, width)
    rows, cols = np.where(union)
    centre_y = 0.5 * (rows.min() + rows.max())
    centre_x = 0.5 * (cols.min() + cols.max())
    extent = max(rows.max() - rows.min(), cols.max() - cols.min()) + 1
    size = int(max(min_size, round(extent * margin)))
    size = min(size, height, width)
    top = int(round(centre_y - size / 2.0))
    left = int(round(centre_x - size / 2.0))
    top = max(0, min(top, height - size))
    left = max(0, min(left, width - size))
    return top, left, size


def save_map(array, path, window=None, cmap=CMAP, vmin=0.0, vmax=1.0):
    """Write one square thumbnail with no axes, padding or interpolation."""

    if window is not None:
        top, left, size = window
        array = array[top:top + size, left:left + size]
    fig = plt.figure(figsize=(SIZE_PX / 100.0, SIZE_PX / 100.0), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()
    ax.imshow(array, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    fig.savefig(path, dpi=100, facecolor="black")
    plt.close(fig)


def pick_distinct_samples(sample_masks, count=3, min_area_frac=0.15):
    """Indices of `count` draws chosen greedily for mutual difference.

    Empty draws and slivers below `min_area_frac` of the largest draw are skipped:
    at thumbnail size they render as blank squares and read as a broken figure
    rather than as a sample. The caption says how many draws were empty, so the
    filtering is stated rather than hidden.
    """

    areas = sample_masks.reshape(len(sample_masks), -1).sum(axis=1)
    floor = min_area_frac * areas.max() if areas.max() > 0 else 0
    candidates = [k for k in range(len(sample_masks)) if areas[k] >= max(1, floor)]
    if len(candidates) <= count:
        return candidates

    def distance(a, b):
        union = np.logical_or(sample_masks[a] > 0, sample_masks[b] > 0).sum()
        inter = np.logical_and(sample_masks[a] > 0, sample_masks[b] > 0).sum()
        return 1.0 - inter / union if union else 0.0

    chosen = [max(candidates, key=lambda k: areas[k])]
    while len(chosen) < count:
        chosen.append(max(
            (k for k in candidates if k not in chosen),
            key=lambda k: min(distance(k, c) for c in chosen)))
    return chosen


def find_index(dataset, patient, stem):
    for index, (image_path, _) in enumerate(dataset.samples):
        if patient in image_path and os.path.basename(image_path) == stem + ".png":
            return index
    raise SystemExit("case {}/{} not found in split".format(patient, stem))

## End-To-End Pipeline

In [ ]:
def train_arm(variant, out_dir):
    out_dir = os.fspath(out_dir)
    latest = os.path.join(out_dir, "latest_checkpoint.pt")
    if os.path.exists(latest) and not FORCE_RETRAIN:
        print("Skipping {}; found {}".format(variant, latest))
        return
    args = SimpleNamespace(
        variant=variant,
        data_root=os.fspath(DATA_ROOT),
        out_dir=out_dir,
        eval_split="val",
        steps=CFG["steps"],
        batch_size=CFG["batch_size"],
        eval_batch_size=CFG["eval_batch_size"],
        crop_size=128,
        max_train=CFG["max_train"],
        max_val=CFG["max_val"],
        num_workers=0,
        device=DEVICE,
        feature_maps=CFG["feature_maps"],
        latent_size=CFG["latent_size"],
        depth=CFG["depth"],
        lr=1e-4,
        lr_final=1e-6,
        lr_decay_steps=5,
        weight_decay=1e-5,
        beta=1.0,
        lambda_disagreement=0.01,
        lambda_alignment=1e-5,
        train_samples=CFG["train_samples"],
        grad_clip=100.0,
        eval_samples=CFG["eval_samples"],
        eval_every=CFG["eval_every"],
        save_every=CFG["save_every"],
        resume=None,
        reset_optimizer=False,
        seed=SEED,
    )
    os.makedirs(args.out_dir, exist_ok=True)
    with open(os.path.join(args.out_dir, "args.json"), "w") as handle:
        json.dump(vars(args), handle, indent=2, sort_keys=True)

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)
    train_loader, val_loader = make_loaders(args)
    model = make_model(args)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    criterion = nn.NLLLoss(reduction="sum")
    start_step = 1
    history_path = os.path.join(args.out_dir, "history.csv")
    best_ged = best_ged_from_history(history_path)

    print("{}: {} train / {} {} samples; writing to {}".format(
        variant, len(train_loader.dataset), len(val_loader.dataset), args.eval_split, args.out_dir))

    train_iter = iter(train_loader)
    start = time.time()
    progress = tqdm(range(start_step, args.steps + 1), desc="train", dynamic_ncols=True,
                    initial=start_step - 1, total=args.steps)
    for step in progress:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
        batch = batch_to_device(batch, args.device)
        row = train_step(model, batch, optimizer, criterion, args, step)
        progress.set_postfix(loss="{:.4f}".format(row["loss"]),
                             seg="{:.4f}".format(row["loss_seg"]),
                             kl="{:.4f}".format(row["loss_kl"]),
                             lr="{:.2e}".format(row["lr"]))
        should_eval = step == 1 or step % args.eval_every == 0 or step == args.steps
        if should_eval:
            metrics = evaluate(model, val_loader, args)
            row.update(metrics)
            row["elapsed_min"] = (time.time() - start) / 60.0
            append_csv(history_path, row)
            tqdm.write("step={step} loss={loss:.4f} dice={val_dice:.4f} iou={val_iou:.4f} ged={val_ged:.4f} unc_mae={val_uncertainty_mae:.4f}".format(**row))
            if best_ged is None or metrics["val_ged"] < best_ged:
                best_ged = metrics["val_ged"]
                save_checkpoint(os.path.join(args.out_dir, "best_checkpoint.pt"), model, optimizer, step, args, metrics)
        if step % args.save_every == 0 or step == args.steps:
            save_checkpoint(os.path.join(args.out_dir, "latest_checkpoint.pt"), model, optimizer, step, args, row)


def evaluate_test_set(runs_dir, variants, out_dir, checkpoint_name="latest_checkpoint.pt"):
    args = SimpleNamespace(
        runs_dir=os.fspath(runs_dir),
        variants=list(variants),
        checkpoint_name=checkpoint_name,
        data_root=os.fspath(DATA_ROOT),
        split="test",
        crop_size=128,
        batch_size=CFG["eval_batch_size"],
        eval_samples=CFG["eval_samples"],
        max_test=CFG["max_test"],
        num_workers=0,
        device=DEVICE,
        out_dir=os.fspath(out_dir),
        seed=SEED,
    )
    os.makedirs(args.out_dir, exist_ok=True)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    dataset = LIDCCrops(root=args.data_root, split=args.split, crop_size=args.crop_size, train=False)
    if args.max_test is not None and args.max_test < len(dataset):
        dataset = Subset(dataset, range(args.max_test))
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False,
                        num_workers=args.num_workers, pin_memory=args.device.startswith("cuda"))
    print("evaluating on {} split: {} images, {} samples each".format(args.split, len(dataset), args.eval_samples))
    results = []
    for variant in args.variants:
        path = os.path.join(args.runs_dir, variant, args.checkpoint_name)
        if not os.path.exists(path):
            print("skipping {}: no checkpoint at {}".format(variant, path))
            continue
        model, checkpoint, saved_variant = load_checkpoint_model(path, args.device)
        metrics = evaluate_checkpoint(model, saved_variant, loader, args)
        metrics["variant"] = saved_variant
        metrics["step"] = int(checkpoint.get("step", -1))
        metrics["lambda_disagreement"] = checkpoint["args"].get("lambda_disagreement")
        metrics["lambda_alignment"] = checkpoint["args"].get("lambda_alignment")
        results.append(metrics)
        print("{}: step {} dice {:.4f} ged {:.4f} unc_corr {:.4f} div_corr {:.4f} nested {:.0%}".format(
            metrics["variant"], metrics["step"], metrics["dice"], metrics["ged"],
            metrics["unc_corr"], metrics["div_corr"], metrics["nested"]))
    if not results:
        raise RuntimeError("no checkpoints evaluated")
    csv_path = os.path.join(args.out_dir, "test_metrics.csv")
    fieldnames = ["variant", "step", "lambda_disagreement", "lambda_alignment"] + [
        k for k in results[0] if k not in ("variant", "step", "lambda_disagreement", "lambda_alignment")]
    with open(csv_path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    control = float(np.nanmean([r["control_corr"] for r in results]))
    lines = [
        "# LIDC {} results ({} samples per image)".format(args.split, args.eval_samples),
        "", "Checkpoints from `{}`.".format(args.runs_dir), "",
        "| variant | step | dice | IoU | GED | U-D corr (entropy) | U-D corr (mutual info) | pred D corr | E[d(S,S')] | nested |",
        "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |",
    ]
    for r in results:
        pred = "n/a" if np.isnan(r["pred_corr"]) else "{:.4f}".format(r["pred_corr"])
        lines.append("| {} | {} | {:.4f} | {:.4f} | {:.4f} | {:.4f} | {:.4f} | {} | {:.4f} | {:.0%} |".format(
            r["variant"], r["step"], r["dice"], r["iou"], r["ged"], r["unc_corr"],
            r["div_corr"], pred, r["pairwise_distance"], r["nested"]))
    lines += ["", "Trivial control correlation: {:.4f}".format(control)]
    md_path = os.path.join(args.out_dir, "summary.md")
    with open(md_path, "w") as handle:
        handle.write("\n".join(lines) + "\n")
    with open(os.path.join(args.out_dir, "args.json"), "w") as handle:
        json.dump(vars(args), handle, indent=2, sort_keys=True)
    print("wrote {} and {}".format(csv_path, md_path))
    return results, md_path


def make_qualitative_figures(baseline_checkpoint, full_checkpoint, out_dir):
    args = SimpleNamespace(
        baseline_checkpoint=os.fspath(baseline_checkpoint),
        full_checkpoint=os.fspath(full_checkpoint),
        data_root=os.fspath(DATA_ROOT),
        split="test",
        out_dir=os.fspath(out_dir),
        num_cases=CFG["figure_cases"],
        indices=None,
        selection="first" if RUN_MODE == "smoke" else "highest-disagreement",
        samples=CFG["figure_samples"],
        device=DEVICE,
        seed=SEED,
    )
    os.makedirs(args.out_dir, exist_ok=True)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    baseline, baseline_checkpoint_obj = load_model(args.baseline_checkpoint, "baseline", args.device)
    full, full_checkpoint_obj = load_model(args.full_checkpoint, "full", args.device)
    baseline_crop = (baseline_checkpoint_obj.get("args") or {}).get("crop_size", 128)
    full_crop = (full_checkpoint_obj.get("args") or {}).get("crop_size", baseline_crop)
    if baseline_crop != full_crop:
        raise ValueError("baseline and full checkpoints use different crop sizes: {} vs {}".format(baseline_crop, full_crop))
    dataset = LIDCCrops(root=args.data_root, split=args.split, crop_size=baseline_crop, train=False, single_random_grader=False)
    cases = choose_cases(dataset, args.indices, args.num_cases, args.selection)
    for case in tqdm(cases, desc="figures", dynamic_ncols=True):
        index = case["index"]
        sample = case["sample"]
        image = sample["image"][None].to(args.device)
        masks = sample["masks"].numpy()
        human_disagreement = compute_disagreement(sample["masks"][None]).numpy()[0, 0]
        baseline_samples, baseline_uncertainty, _ = sample_model(baseline, image, args.samples, has_disagreement_head=False)
        full_samples, full_uncertainty, predicted_disagreement = sample_model(full, image, args.samples, has_disagreement_head=True)
        out_name = "case_{:04d}_{}_{}.png".format(index, case["patient"], case["stem"])
        out_path = os.path.join(args.out_dir, out_name)
        make_case_figure(out_path, "{} index {} / {} / {}".format(args.split, index, case["patient"], case["stem"]),
                         sample["image"][0].numpy(), masks, human_disagreement,
                         baseline_samples, baseline_uncertainty, full_samples, full_uncertainty, predicted_disagreement)
        print("wrote", out_path)


def make_architecture_panels(checkpoint, out_dir):
    args = SimpleNamespace(
        checkpoint=os.fspath(checkpoint),
        data_root=os.fspath(DATA_ROOT),
        split="test",
        patient="LIDC-IDRI-0217",
        stem="z-129.0_c0",
        out_dir=os.fspath(out_dir),
        samples=max(16, CFG["eval_samples"]),
        zoom_margin=1.8,
        min_area_frac=0.15,
        device=DEVICE,
        seed=SEED,
    )
    os.makedirs(args.out_dir, exist_ok=True)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    dataset = LIDCCrops(root=args.data_root, split=args.split, crop_size=128, train=False)
    sample = dataset[find_index(dataset, args.patient, args.stem)]
    image = sample["image"][None].to(args.device)
    masks = sample["masks"][None].to(args.device)
    checkpoint_obj = torch.load(args.checkpoint, map_location=args.device, weights_only=False)
    saved = dict(checkpoint_obj["args"])
    saved["device"] = args.device
    model = make_model(SimpleNamespace(**saved))
    model.load_state_dict(checkpoint_obj["model_state_dict"])
    model.eval()
    with torch.no_grad():
        model.reset()
        prediction = model(image)
        predicted_mask = torch.argmax(prediction, dim=1)[0].cpu().numpy()
        outputs = torch.stack(model.sample_prior(args.samples, out_device=image.device, input_=image), dim=0)
        uncertainty = model_uncertainty_from_samples(outputs)[0, 0].cpu().numpy()
        predicted_disagreement = model.predict_disagreement()[0, 0].cpu().numpy()
        sample_masks = torch.argmax(outputs, dim=2)[:, 0].cpu().numpy()
        sample_probs = torch.softmax(outputs, dim=2)[:, 0, 1].cpu().numpy()
    human = compute_disagreement(masks)[0, 0].cpu().numpy()
    grader_masks = sample["masks"].numpy()
    window = lesion_window(grader_masks, margin=args.zoom_margin)
    save_map(sample["image"][0].numpy(), os.path.join(args.out_dir, "ct.png"), window, cmap="gray")
    for grader in range(masks.shape[1]):
        save_map(grader_masks[grader], os.path.join(args.out_dir, "mask{}.png".format(grader)), window, cmap="gray")
    save_map(human, os.path.join(args.out_dir, "dgt.png"), window)
    save_map(uncertainty, os.path.join(args.out_dir, "u.png"), window)
    save_map(predicted_disagreement, os.path.join(args.out_dir, "dhat.png"), window)
    save_map(predicted_mask, os.path.join(args.out_dir, "pred.png"), window, cmap="gray")
    chosen = pick_distinct_samples(sample_masks, count=3, min_area_frac=args.min_area_frac)
    for slot, k in enumerate(chosen):
        save_map(sample_probs[k], os.path.join(args.out_dir, "sample{}.png".format(slot)), window, cmap="gray")
    print("wrote {}".format(args.out_dir))

## Data

In [ ]:
download_lidc(DATA_ROOT)

## Human Disagreement Preview

In [ ]:
preview_dir = OUTPUT_ROOT / "preview"
preview_dir.mkdir(parents=True, exist_ok=True)
ds_preview = LIDCCrops(root=os.fspath(DATA_ROOT), split="val", train=False, single_random_grader=False)
sample = ds_preview[0]
d_map = compute_disagreement(sample["masks"][None]).numpy()[0, 0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.4))
panels = [sample["image"][0].numpy()] + [sample["masks"][i].numpy() for i in range(4)] + [d_map]
titles = ["image", "mask 1", "mask 2", "mask 3", "mask 4", "human D"]
for ax, panel, title in zip(axes, panels, titles):
    cmap = "magma" if title == "human D" else "gray"
    ax.imshow(panel, cmap=cmap, vmin=0 if cmap == "magma" else None, vmax=1 if cmap == "magma" else None)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
preview_path = preview_dir / "disagreement_preview.png"
fig.savefig(preview_path, dpi=160)
plt.close(fig)
display(DisplayImage(filename=str(preview_path)))

## Arm A-C: ELBO Ablation

In [ ]:
fcomb_runs = OUTPUT_ROOT / "fcombfix" / "lidc_ablation"
for variant in MAIN_VARIANTS:
    train_arm(variant, fcomb_runs / variant)

## Test Results

In [ ]:
fcomb_eval = OUTPUT_ROOT / "fcombfix" / "final_eval_100k"
_, summary_path = evaluate_test_set(fcomb_runs, MAIN_VARIANTS, fcomb_eval, checkpoint_name="latest_checkpoint.pt")
display(Markdown(Path(summary_path).read_text()))


## Qualitative Results

In [ ]:
fig_dir = OUTPUT_ROOT / "fcombfix" / "figures_100k"
make_qualitative_figures(fcomb_runs / "baseline" / "latest_checkpoint.pt", fcomb_runs / "full" / "latest_checkpoint.pt", fig_dir)
figure_paths = sorted(fig_dir.glob("case_*.png"))
print(f"Generated {len(figure_paths)} qualitative figures in {fig_dir}")
for path in figure_paths[: min(4, len(figure_paths))]:
    print(path)
    display(DisplayImage(filename=str(path)))

## Architecture Panels

In [ ]:
asset_dir = OUTPUT_ROOT / "overleaf_figs"
make_architecture_panels(fcomb_runs / "full" / "latest_checkpoint.pt", asset_dir)
for name in ["ct.png", "mask0.png", "mask1.png", "mask2.png", "mask3.png", "dgt.png", "sample0.png", "sample1.png", "sample2.png", "u.png", "dhat.png", "pred.png"]:
    path = asset_dir / name
    if path.exists():
        print(path)
        display(DisplayImage(filename=str(path), width=120))

## Generated Artifacts

- `fcombfix/lidc_ablation/<baseline|head|full>/history.csv`
- `fcombfix/lidc_ablation/<baseline|head|full>/{best,latest}_checkpoint.pt`
- `fcombfix/final_eval_100k/{test_metrics.csv,summary.md}`
- `fcombfix/figures_100k/case_*.png`
- `overleaf_figs/*.png`